# Sailboat pricing — question-by-question workbench

**Synthetic demo only until DATA_PATH is set.**

- [00 — What this project answers](#section-00)
- [01 — Setup and editable settings](#section-01)
- [02 — Map columns and load data](#section-02)
- [03 — Part 1 Q1: numeric extraction and units](#section-03)
- [04 — Part 1 Q2: categorical cleanup and consistency](#section-04)
- [05 — Part 1 Q1a: Beneteau mainsail and category draft](#section-05)
- [06 — Part 1 Q3: three largest feature correlations](#section-06)
- [07 — Listing-price distribution and expensive boats](#section-07)
- [08 — Geographic pricing and consistency](#section-08)
- [09 — Mean and median by build year](#section-09)
- [10 — Part 2: target, features and unseen-make splits](#section-10)
- [11 — Baselines and pricing models](#section-11)
- [12 — Compare make and variant validation](#section-12)
- [13 — Residuals, segments, robustness and feature importance](#section-13)
- [14 — Freeze and reserved unseen-make evaluation](#section-14)
- [15 — Predict ten boats and report average/median](#section-15)
- [16 — Write the conclusions and next steps](#section-16)

<a id="section-00"></a>

## 00 — What this project answers

This is a complete, adaptable solution skeleton for the supplied sailboat questions. **No real dataset was supplied. All default data and any default outputs are synthetic demonstrations.** Run with your CSV to obtain numerical answers; do not present demo results as Kaggle findings.

Navigation: change settings in 01, replace the demo using DATA_PATH in 02, inspect parsing/audit in 03–04, run Part 1 in 05–09 and Part 2 in 10–16. Every section is matched in the guide and notebook. The code supports Python 3.8 syntax and both older/newer OneHotEncoder APIs.

Assumptions to verify: each row is a listing; target is already converted to USD; dimensions are metric unless explicit units say otherwise; 'model' means hull type only if the schema confirms it; 'main sale area' means mainsail area; the ten boats are supplied separately, otherwise ten held-out listings illustrate predictions. The phrase 'advanced, average and median' is ambiguous: this solution reports individual predictions, total, average and median; no unidentified 'advanced' statistic is invented.

Part 1 is a full-dataset descriptive exercise. Part 2 uses make/variant-aware validation, but if you adapt model choices after looking at full-data Part 1, the reserved holdout is **not truly blind**. For a strict research protocol do Part 1 on development data and get a separate final evaluation dataset.

Requirements: `python -m pip install numpy pandas scipy matplotlib scikit-learn statsmodels nbformat jupyter`. No uploads, APIs, or network calls are needed during analysis. Keep real data, trained artifacts and prediction files outside this public repository.

<a id="section-01"></a>

## 01 — Setup and editable settings

Set DATA_PATH to your CSV and TEN_BOATS_PATH to the ten unpriced boats if supplied. Units for bare numbers must come from the dataset documentation; never guess from magnitude. REFERENCE_YEAR should be the listing/valuation year, not automatically today's year for a historical dataset. If listings span years, compute age using a row-specific listing-year column. Locale DECIMAL_MARK='.' handles 1,234.5; choose ',' for 1.234,5. An isolated comma such as 1,234 is ambiguous unless locale is confirmed.

In [ ]:
import re, unicodedata, inspect, copy, io, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from IPython.display import display
SEED = 42
DATA_PATH = None                   # e.g. 'boats.csv'; None runs a DEMO
TEN_BOATS_PATH = None              # e.g. 'ten_boats.csv'
REFERENCE_YEAR = 2026              # CHANGE to valuation/listing year
DECIMAL_MARK = '.'                 # ',' for documented comma-decimal locale
MIN_PAIR_N = 30                    # minimum complete pairs for correlations
EXPENSIVE_QUANTILE = .95           # define expensive as top 5% of valid prices
MIN_CATEGORY_N = 5                 # still show all categories for the question
RUN_REGION_ADJUSTMENT = True       # optional statsmodels descriptive regression
RUN_HOLDOUT = False                # enable after freezing choices
RUN_IMPORTANCE = True
RUN_ABLATION = False
RUN_SAVE = False                   # local outputs only
plt.rcParams.update({'figure.figsize':(8,4),'font.size':10,'axes.grid':True})

<a id="section-02"></a>

## 02 — Map columns and load data

Aliases cover the spoken column names. If an exact name is missed, add COLUMN_OVERRIDES, e.g. {'your exact column': 'hull_type'}. Duplicate mappings raise an error so nothing is silently overwritten. Preserve the raw frame. Review the printed map before analysis. Hull type/model/variant can mean different things in actual files; edit the map after looking at their values.

In [ ]:
def key(value):
    value = unicodedata.normalize('NFKD', str(value)).encode('ascii','ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+','_',value).strip('_')

ALIASES = {
 'make':['make','manufacturer','milk','r_milk'],
 'variant':['variant','boat_model','model_name'],
 'geographic_region':['geographic_region','geographical_region'],
 'location':['country_region_state','country_region','country','location'],
 'price_usd':['listing_price_usd','listing_price','price','price_usd','listing_price_in_usd'],
 'year_built':['year','year_built','built_year'],
 'hull_type':['model','hull_type'], 'category':['category'],
 'designer':['sailboat_designer','designer'], 'construction':['construction','material','materials'],
 'last_built_year':['last_built_hull','last_built','last_built_year'],
 'appendages':['appendages'], 'hull_length_m':['hull_length','hull_length_m','length'],
 'beam_m':['beam','beam_m'], 'draft_m':['draft','draft_m'],
 'rigging_type':['rigging_type','rigging'],
 'upwind_sail_area_m2':['upwind_sail_area','upwind_sail_area_m2'],
 'downwind_sail_area_m2':['downwind_sail_area','downwind_sail_area_m2'],
 'mainsail_area_m2':['mainsail_area','main_sail_area','main_sale_area','mainsail_area_m2'],
 'engine_power_kw':['engine_power','engines_power','engine_s_power','engine_power_kw'],
 'light_displacement_kg':['light_displacement','displacement','light_displacement_kg'],
 'headroom_m':['maximum_headroom','max_headroom','headroom','headroom_m'],
 'deck_area_m2':['deck_area','deck_area_m2']}
COLUMN_OVERRIDES = {}  # exact source name -> canonical name
alias_lookup = {key(alias):name for name,aliases in ALIASES.items() for alias in aliases}
def canonicalize(frame):
    mapping = {c:COLUMN_OVERRIDES.get(c,alias_lookup.get(key(c),key(c))) for c in frame.columns}
    result = frame.rename(columns=mapping).copy()
    assert result.columns.is_unique, 'Multiple source columns map to one field; revise aliases'
    return result, mapping

if DATA_PATH is not None:
    raw = pd.read_csv(DATA_PATH)
else:
    # SYNTHETIC example ONLY: prices below are fabricated for testing code.
    rng = np.random.RandomState(SEED)
    n = 480
    make_idx = rng.randint(0,16,n)
    length = rng.uniform(7,22,n)
    beam = length*.27 + rng.uniform(-.2,.6,n)
    year = rng.randint(1975,2025,n)
    regions = rng.choice(['europe','north america','caribbean'],n)
    price = np.exp(9 + .16*length + .015*(year-1990) + .10*(regions=='europe') + rng.normal(0,.3,n))
    raw = pd.DataFrame({'Make':np.where(make_idx==0,'Bénéteau',['make_'+str(i) for i in make_idx]),
        'Variant':['series_'+str(i%3) for i in range(n)],'Geographic Region':regions,
        'Country/Region/State':rng.choice(['france','usa','greece'],n),
        'Listing Price (USD)':['$'+format(v,',.2f') for v in price], 'Year':year,
        'Model':rng.choice(['monohull','catamaran'],n), 'Category':rng.choice(['cruiser','racer'],n),
        'Sailboat Designer':'designer', 'Construction':'fiberglass', 'Last Built Hull':2025,
        'Appendages':'fin keel', 'Hull Length':[format(v,'.2f')+' m' for v in length],
        'Beam':[format(v,'.2f')+' m' for v in beam], 'Draft':[format(v,'.2f')+' m' for v in length*.12],
        'Rigging Type':'sloop', 'Upwind Sail Area':[format(v,'.1f')+' m²' for v in length**2*.5],
        'Downwind Sail Area':[format(v,'.1f')+' m²' for v in length**2*.8],
        'Mainsail Area':[format(v,'.1f')+' m²' for v in length**2*.3],
        "Engine's Power":['2 x 20 hp' if i%2 else '25 kW' for i in range(n)],
        'Light Displacement':[format(v,'.1f')+' kg' for v in length**3*4],
        'Maximum Headroom':'1.90 m','Deck Area':length*beam})
    raw.loc[0,'Draft']='5 ft 3 in'
    raw.loc[1,'Mainsail Area']='not available'
    print('SYNTHETIC DEMO: replace DATA_PATH for actual answers')
data, column_map = canonicalize(raw)
display(pd.Series(column_map,name='canonical_column').to_frame())
assert {'make','variant','price_usd','year_built'}.issubset(data.columns)
display(data.head(3))

<a id="section-03"></a>

## 03 — Part 1 Q1: numeric extraction and units

Use dimensional parsing, not 'strip every non-digit': that would turn '10.5 m / 34 ft' into nonsense. Prefer explicitly metric representations, then convert feet/inches, square feet, tonnes/lb and hp/CV to m, m², kg and kW. Twin-engine power is summed only when written explicitly as a multiplier. Values with ranges, inequalities, conflicting equivalent units or several ambiguous numbers are flagged, not silently averaged. Bare numeric strings use only configured documented units. This recipe cannot infer the meaning of every free-text description; examine the audit and extend exact patterns as required. Comma decimals require confirmed locale. Units in column headers must be incorporated into the configuration/mapping.

In [ ]:
UNIT_DEFAULT = {
 'price_usd':'usd','hull_length_m':'m','beam_m':'m','draft_m':'m','headroom_m':'m',
 'upwind_sail_area_m2':'m2','downwind_sail_area_m2':'m2','mainsail_area_m2':'m2',
 'deck_area_m2':'m2','engine_power_kw':'kw','light_displacement_kg':'kg'}
# CHANGE defaults if the source documents feet, hp, tonnes, etc.
FACTORS = {'usd':1,'m':1,'cm':.01,'ft':.3048,'in':.0254,
           'm2':1,'ft2':.09290304,'kg':1,'t':1000,'lb':.45359237,
           'kw':1,'hp':.745699872,'cv':.73549875}
FAMILY = {'usd':'price','m':'length','cm':'length','ft':'length','in':'length',
          'm2':'area','ft2':'area','kg':'mass','t':'mass','lb':'mass','kw':'power','hp':'power','cv':'power'}
NUMBER = r'[-+]?(?:\d[\d.,\s]*\d|\d)(?:[eE][-+]?\d+)?'
MISSING = {'','nan','none','null','n/a','na','-','unknown','not available','price on request','poa'}
def read_number(token):
    token = re.sub(r'\s+','',token)
    if DECIMAL_MARK == '.':
        # commas are documented thousands separators
        if ',' in token and not re.fullmatch(r'[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?(?:[eE][-+]?\d+)?',token):
            return np.nan
        token = token.replace(',','')
    else:
        token = token.replace('.','').replace(',','.')
    try: return float(token)
    except ValueError: return np.nan

def parse_quantity(value, default):
    if pd.isna(value): return np.nan,'missing'
    if isinstance(value,(int,float,np.number)):
        return (float(value)*FACTORS[default],'bare:'+default) if np.isfinite(value) else (np.nan,'nonfinite')
    s = str(value).strip().lower().replace('\u00a0',' ').replace('²','2')
    if s in MISSING: return np.nan,'missing'
    if re.search(r'[<>~]|\b(?:from|between|approx)\b|\d\s*[-–]\s*\d',s):
        return np.nan,'ambiguous_range_or_bound'
    s = re.sub(r'\bsq\.?\s*(?:ft|feet)\b','ft2',s)
    s = re.sub(r'\bsq\.?\s*(?:m|meters|metres)\b','m2',s)
    s = re.sub(r'\bsquare\s+(?:feet|foot)\b','ft2',s)
    s = re.sub(r'\bsquare\s+(?:meters|metres)\b','m2',s)
    s = s.replace('m^2','m2').replace('ft^2','ft2')
    s = s.replace("'",' ft ').replace('"',' in ')
    s = re.sub(r'\b(?:us\$|usd)\b',' usd ',s).replace('$',' usd ')
    if re.search(r'\btons?\b',s): return np.nan,'ambiguous_ton_unit'
    if FAMILY[default]=='price':
        if re.search(r'€|£|eur|gbp|cad|aud|\bkr\b',s): return np.nan,'wrong_currency'
        q = re.fullmatch(r'\s*(?:usd\s*)?('+NUMBER+r')\s*(?:usd)?\s*',s)
        if not q: return np.nan,'unparsed_price'
        v=read_number(q.group(1))
        return (v,'usd') if np.isfinite(v) else (np.nan,'ambiguous_numeric_locale')
    units = r'm2|ft2|kw|hp|cv|kg|lbs?|tonnes?|tons?|feet|foot|ft|inches|inch|in|centimeters|centimetres|cm|meters|metres|m|t'
    aliases = {'lbs':'lb','tonne':'t','tonnes':'t','ton':'t','tons':'t','feet':'ft','foot':'ft','inches':'in','inch':'in','meters':'m','metres':'m','centimeters':'cm','centimetres':'cm'}
    multiplier = re.search(r'\b(\d+)\s*[x×]\s*',s)
    if multiplier and FAMILY[default]!='power': return np.nan,'unexpected_multiplier'
    s_match = re.sub(r'\b\d+\s*[x×]\s*','',s) if multiplier else s
    matches = [(read_number(a),aliases.get(u,u)) for a,u in re.findall('('+NUMBER+r')\s*('+units+r')\b',s_match)]
    relevant = [(v,u) for v,u in matches if FAMILY[u]==FAMILY[default]]
    if relevant:
        if not all(np.isfinite(v) for v,u in relevant): return np.nan,'ambiguous_numeric_locale'
        # feet + inches is one dimensional value, not two competing values
        if len(relevant)==2 and {u for _,u in relevant}=={'ft','in'}:
            return sum(v*FACTORS[u] for v,u in relevant),'ft+in'
        converted = [v*FACTORS[u] for v,u in relevant]
        preferred = [v*FACTORS[u] for v,u in relevant if u==default]
        result = preferred[0] if preferred else converted[0]
        if len(converted)>1 and not np.allclose(converted,result,rtol=.03,atol=.01):
            return np.nan,'conflicting_or_multiple_values'
        if multiplier: result *= int(multiplier.group(1))
        return result,'explicit:'+relevant[0][1]
    bare = re.fullmatch(r'\s*('+NUMBER+r')\s*',s)
    if bare:
        v=read_number(bare.group(1))
        return (v*FACTORS[default],'bare:'+default) if np.isfinite(v) else (np.nan,'ambiguous_numeric_locale')
    return np.nan,'unparsed'

def parse_year(value):
    if pd.isna(value) or str(value).strip().lower() in MISSING: return np.nan,'missing'
    s = str(value).strip()
    if re.fullmatch(r'\d{4}(?:\.0)?',s):
        return float(s),'year'
    years = re.findall(r'(?<!\d)(?:18|19|20)\d{2}(?!\d)',s)
    if len(years)==1 and not re.search(r'[-–/]|\b(?:since|present|ongoing)\b',s.lower()):
        return float(years[0]),'year_in_text'
    return np.nan,'ambiguous_year'  # ongoing production is not a final build year

def clean_frame(frame):
    result = frame.copy(); records=[]
    for c,u in UNIT_DEFAULT.items():
        if c not in result: result[c]=np.nan
        parsed = result[c].map(lambda v:parse_quantity(v,u))
        for i,(v,status) in parsed.items(): records.append({'row':i,'column':c,'raw':result.loc[i,c],'value':v,'status':status})
        result[c] = parsed.map(lambda z:z[0])
    for c in ['year_built','last_built_year']:
        if c not in result: result[c]=np.nan
        parsed = result[c].map(parse_year)
        for i,(v,status) in parsed.items(): records.append({'row':i,'column':c,'raw':result.loc[i,c],'value':v,'status':status})
        result[c] = parsed.map(lambda z:z[0])
    return result,pd.DataFrame(records)
data, parse_audit = clean_frame(data)
display(parse_audit.groupby(['column','status']).size().rename('count').reset_index())
display(parse_audit[parse_audit.status.str.contains('ambiguous|conflicting|wrong|unparsed|nonfinite')].head(30))

<a id="section-04"></a>

## 04 — Part 1 Q2: categorical cleanup and consistency

Normalize whitespace/case/accents, unify only known equivalent labels and preserve the raw source. 'Beneteau' and 'Bénéteau' become the same make. Do not fuzzy-merge manufacturers or variants without review. Check impossible years, nonpositive physical measurements, bad currencies, suspicious ratios and duplicate listings. Flag rather than deleting plausible luxury boats. Target values that are nonfinite/nonpositive cannot be modeled as positive USD prices. Exact duplicates are removed with a log; listings sharing make/variant/specifications are not automatically duplicates. If a listing ID exists, check/reconcile duplicates by ID. Missing category/specification values remain missing for Part 1 (denominators reported), then are imputed within training folds for modeling.

Deck area here is length × beam, a rectangular-envelope proxy; verify an existing column against the stated definition. If inconsistent, use the recomputed value and retain the discrepancy. This deterministic feature is not independent of length and beam and may dominate correlation rankings mechanically. Last-built-year is model-level metadata; future archive information may be unavailable at listing time, so it is excluded from the predictive default.

In [ ]:
CAT_COLS = ['make','variant','geographic_region','location','hull_type','category','designer','construction','appendages','rigging_type']
def clean_category(v):
    if pd.isna(v): return np.nan
    s = unicodedata.normalize('NFKD',str(v)).encode('ascii','ignore').decode().lower()
    s = re.sub(r'\s+',' ',s).strip()
    return np.nan if s in MISSING else s
def clean_categories(frame):
    out=frame.copy()
    for c in CAT_COLS:
        if c not in out: out[c]=np.nan
        out[c]=out[c].map(clean_category)
    out['hull_type']=out['hull_type'].replace({'mono hull':'monohull','mono-hull':'monohull','cat':'catamaran'})
    out['construction']=out['construction'].replace({'fibre glass':'fiberglass','fibreglass':'fiberglass','grp':'fiberglass'})
    return out
data = clean_categories(data)
quality_flags=[]
for c in UNIT_DEFAULT:
    bad = data[c].notna() & ((~np.isfinite(data[c])) | (data[c]<=0))
    for i in data.index[bad]: quality_flags.append({'row':i,'field':c,'issue':'nonpositive_or_nonfinite','value':data.loc[i,c]})
    data.loc[bad,c]=np.nan
for c in ['year_built','last_built_year']:
    bad = data[c].notna() & ((data[c]<1800) | (data[c]>REFERENCE_YEAR) | (data[c]%1 != 0))
    for i in data.index[bad]: quality_flags.append({'row':i,'field':c,'issue':'invalid_year','value':data.loc[i,c]})
    data.loc[bad,c]=np.nan
data['deck_area_given_m2']=data['deck_area_m2']
computed_deck = data.hull_length_m * data.beam_m
deck_bad = computed_deck.notna() & data.deck_area_m2.notna() & ~np.isclose(computed_deck,data.deck_area_m2,rtol=.02,atol=.05)
print('Deck-area inconsistencies:',int(deck_bad.sum()))
data['deck_area_m2']=computed_deck
ratio = data.beam_m/data.hull_length_m
print('Rows with beam >= length (review):',int((ratio>=1).sum()))
duplicates = data.duplicated(subset=[c for c in data if c!='deck_area_given_m2'])
print('Exact cleaned duplicates removed:',int(duplicates.sum()))
data = data.loc[~duplicates].copy().reset_index(drop=True)
data['row_id']=np.arange(len(data))
display(pd.DataFrame(quality_flags).head(30))
display(pd.DataFrame({'missing_fraction':data.isna().mean(),'distinct':data.nunique()}).sort_values('missing_fraction',ascending=False))
print('Rows without valid USD target:',data.price_usd.isna().sum())

<a id="section-05"></a>

## 05 — Part 1 Q1a: Beneteau mainsail and category draft

Compute the mean mainsail area among valid Beneteau listings; report total listings, observed count and units. Because specifications repeat across listings, also show an equally weighted model-variant estimate to expose sampling bias. Answer the category question using the largest observed median draft, show all ties, counts and missingness; a second table with a minimum count is a robustness check, not a replacement for the literal answer. No category or Beneteau value can be inferred without the real rows.

In [ ]:
bene = data.loc[data.make.eq('beneteau')]
beneteau_answer = pd.Series({'listings':len(bene),'valid_mainsail_count':bene.mainsail_area_m2.notna().sum(),
                            'mean_mainsail_area_m2':bene.mainsail_area_m2.mean(),
                            'mean_across_variant_means_m2':bene.groupby('variant').mainsail_area_m2.mean().mean()})
display(beneteau_answer)
draft_by_category = data.groupby('category',dropna=False).draft_m.agg(['size','count','median']).rename(columns={'count':'valid_count','median':'median_draft_m'})
display(draft_by_category.sort_values('median_draft_m',ascending=False))
valid_category = draft_by_category[draft_by_category.index.notna()].dropna(subset=['median_draft_m'])
if len(valid_category):
    highest = valid_category.median_draft_m.max()
    print('Highest-median category/categories:')
    display(valid_category[np.isclose(valid_category.median_draft_m,highest)])
display(draft_by_category[draft_by_category.valid_count>=MIN_CATEGORY_N].sort_values('median_draft_m',ascending=False))

<a id="section-06"></a>

## 06 — Part 1 Q3: three largest feature correlations

Rank unique numerical-feature pairs by absolute Pearson correlation; show signs and complete-pair counts, then compare Spearman ranking and scatter plots. Exclude target, row IDs and duplicated deck-area source values from feature-pair ranking; price associations are a separate table. Include the derived deck proxy for the literal feature question and show a second ranking without it. Require enough paired observations, examine top-three coverage and missingness. Strong scale relationships (length/displacement/sail area/beam) are physically plausible hypotheses, not verified rankings. Repeat at make+variant level as a robustness check because multiple listings of the same model can dominate. Avoid causal claims and avoid claiming top-three certainty when sample correlations are nearly tied.

In [ ]:
NUMERIC_FEATURES = list(UNIT_DEFAULT)
NUMERIC_FEATURES.remove('price_usd')
NUMERIC_FEATURES += ['year_built','last_built_year']
def rank_pairs(frame, cols, method='pearson', minimum=MIN_PAIR_N):
    correlation = frame[cols].corr(method=method)
    rows=[]
    for i,a in enumerate(cols):
        for b in cols[i+1:]:
            n=frame[[a,b]].dropna().shape[0]
            r=correlation.loc[a,b]
            if n>=minimum and np.isfinite(r): rows.append({'feature_1':a,'feature_2':b,'correlation':r,'abs_correlation':abs(r),'n_pairs':n})
    return pd.DataFrame(rows,columns=['feature_1','feature_2','correlation','abs_correlation','n_pairs']).sort_values('abs_correlation',ascending=False)
pearson_pairs = rank_pairs(data,NUMERIC_FEATURES)
spearman_pairs = rank_pairs(data,NUMERIC_FEATURES,'spearman')
display(pearson_pairs.head(3),spearman_pairs.head(3))
display(rank_pairs(data,[c for c in NUMERIC_FEATURES if c!='deck_area_m2']).head(3))
for _,pair in pearson_pairs.head(3).iterrows():
    a,b=pair.feature_1,pair.feature_2
    z=data[[a,b]].dropna()
    plt.scatter(z[a],z[b],alpha=.3,s=10)
    plt.xlabel(a); plt.ylabel(b); plt.title('Pearson r=%.3f, n=%d'%(pair.correlation,pair.n_pairs)); plt.tight_layout(); plt.show()
display(data[[c for c in NUMERIC_FEATURES if data[c].nunique()>1]].corrwith(data.price_usd).sort_values(ascending=False).rename('correlation_with_price'))
model_specs = data.groupby(['make','variant'])[NUMERIC_FEATURES].median()
display(rank_pairs(model_specs,NUMERIC_FEATURES,minimum=min(10,MIN_PAIR_N)).head(3))

<a id="section-07"></a>

## 07 — Listing-price distribution and expensive boats

Plot prices in dollars and log dollars (positive values only); report percentiles. Define expensive as the top 5% of valid asking prices, report the dollar threshold and sensitivity to top 10%/1%. Describe numeric medians and categorical prevalence versus the remainder, with counts and prevalence ratios. This is descriptive selection by target, not an unbiased discovery of pricing causes. Expensive boats may be larger, newer, multihulls or from premium makes; check these hypotheses rather than asserting them. Means are sensitive to very large yachts; medians and log distributions make concentration clearer.

In [ ]:
priced = data.loc[data.price_usd.notna()].copy()
assert len(priced)>=20, 'Too few valid prices for this default workflow'
display(priced.price_usd.describe(percentiles=[.01,.05,.5,.9,.95,.99]))
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].hist(priced.price_usd,bins=40); ax[0].set(xlabel='Asking price (USD)',ylabel='Listings',title='Price distribution')
ax[1].hist(np.log10(priced.price_usd),bins=35); ax[1].set(xlabel='log10 asking price (USD)',ylabel='Listings',title='Price distribution on log scale')
plt.tight_layout(); plt.show()
expensive_threshold = priced.price_usd.quantile(EXPENSIVE_QUANTILE)
priced['expensive']=priced.price_usd>=expensive_threshold
print('Expensive threshold USD:',expensive_threshold,'| count:',int(priced.expensive.sum()))
display(priced.groupby('expensive')[NUMERIC_FEATURES].median().T)
for c in ['make','hull_type','category','construction','geographic_region']:
    counts = pd.crosstab(priced[c].fillna('__MISSING__'),priced.expensive).reindex(columns=[False,True],fill_value=0)
    counts.columns=['other_n','expensive_n']
    counts['expensive_share']=counts.expensive_n/max(counts.expensive_n.sum(),1)
    counts['other_share']=counts.other_n/max(counts.other_n.sum(),1)
    counts['prevalence_ratio']=counts.expensive_share/counts.other_share.replace(0,np.nan)
    display(counts.sort_values('expensive_n',ascending=False).head(12))
display(pd.DataFrame([{'quantile':q,'threshold_usd':priced.price_usd.quantile(q),'n':int((priced.price_usd>=priced.price_usd.quantile(q)).sum())} for q in [.9,.95,.99]]))

<a id="section-08"></a>

## 08 — Geographic pricing and consistency

Start with counts/mean/median and box plots on log prices. Then compare like with like: region composition can differ in age, size, hull type and make/variant. Estimate a descriptive log-price regression controlling for physical characteristics, age, hull type and make+variant fixed effects; cluster standard errors by make+variant. The region coefficient exp(beta)-1 is an adjusted conditional association relative to the reference region, not a causal premium. Verify that comparable models span regions; disconnected/thin support and country-region collinearity make effects unidentified or unstable. Do not include country and region simultaneously without a clear design.

Consistency checks: same-variant cross-region comparisons; region ratios within size/age/hull strata with minimum counts; cluster bootstrap of raw regional medians. Stability of sign/magnitude across these is evidence of consistency. No consistency claim is guaranteed. Cluster bootstrap assumes model families are reasonable independent units; few groups need caution. Region can proxy condition, taxes, seller mix, equipment, currency conversion and listing dates. These omitted variables limit interpretation.

In [ ]:
region_summary = priced.groupby('geographic_region').price_usd.agg(['count','mean','median','std']).sort_values('median',ascending=False)
display(region_summary)
region_data=[np.log(priced.loc[priced.geographic_region.eq(r),'price_usd']) for r in region_summary.index]
if region_data:
    plt.boxplot(region_data,labels=region_summary.index); plt.xticks(rotation=30,ha='right')
    plt.ylabel('log asking price (USD)'); plt.title('Unadjusted prices by region'); plt.tight_layout(); plt.show()
priced['variant_key']=priced.make.fillna('__MISSING__')+'::'+priced.variant.fillna('__MISSING__')
overlap = priced.groupby('variant_key').geographic_region.nunique()
print('Variants represented in >=2 regions:',int((overlap>=2).sum()),'of',len(overlap))
same_variant = priced[priced.variant_key.isin(overlap[overlap>=2].index)].pivot_table(index='variant_key',columns='geographic_region',values='price_usd',aggfunc='median')
display(same_variant.head(15))
priced['size_bin']=pd.qcut(priced.hull_length_m,4,duplicates='drop')
priced['age_bin']=pd.cut(REFERENCE_YEAR-priced.year_built,[-1,10,25,50,np.inf])
strata = priced.groupby(['hull_type','size_bin','age_bin','geographic_region'],observed=True).price_usd.agg(['count','median'])
display(strata[strata['count']>=MIN_CATEGORY_N].head(30))
if RUN_REGION_ADJUSTMENT:
    try:
        import statsmodels.formula.api as smf
        reg=priced.dropna(subset=['price_usd','hull_length_m','beam_m','year_built','hull_type','geographic_region','make','variant']).copy()
        reg['log_price']=np.log(reg.price_usd)
        reg['age']=REFERENCE_YEAR-reg.year_built
        reg['log_length']=np.log(reg.hull_length_m); reg['log_beam']=np.log(reg.beam_m)
        enough=reg.variant_key.nunique()>=10 and reg.geographic_region.nunique()>=2 and (overlap>=2).sum()>=5
        if enough:
            formula='log_price ~ C(geographic_region) + log_length + log_beam + age + I(age**2) + C(hull_type) + C(variant_key)'
            fit=smf.ols(formula,reg).fit(cov_type='cluster',cov_kwds={'groups':reg.variant_key})
            print('Reference region:',sorted(reg.geographic_region.unique())[0])
            terms=[t for t in fit.params.index if 'C(geographic_region)' in t]
            adjusted_region=pd.DataFrame({'log_coef':fit.params[terms],'se_cluster':fit.bse[terms],
                'association_pct':100*np.expm1(fit.params[terms]),
                'low_pct':100*np.expm1(fit.conf_int().loc[terms,0]),'high_pct':100*np.expm1(fit.conf_int().loc[terms,1])})
            display(adjusted_region)
            design=fit.model.exog
            print('Design rank / columns:',np.linalg.matrix_rank(design),design.shape[1])
            if np.linalg.matrix_rank(design)<design.shape[1]: print('CAUTION: collinear fixed effects; interpret only identified contrasts')
        else: print('Insufficient model-region overlap/groups for the proposed adjustment')
    except ImportError: print('Install statsmodels to run adjusted regional analysis')

# Cluster bootstrap raw regional medians, preserving whole model families.
random=np.random.RandomState(SEED); group_ids=priced.variant_key.unique(); bootstrap=[]
if len(group_ids)>=10:
    for _ in range(150):
        sampled=random.choice(group_ids,size=len(group_ids),replace=True)
        parts=[priced[priced.variant_key.eq(g)] for g in sampled]
        bootstrap.append(pd.concat(parts).groupby('geographic_region').price_usd.median())
    boot=pd.DataFrame(bootstrap)
    display(pd.DataFrame({'median':region_summary['median'],'cluster_boot_low':boot.quantile(.025),'cluster_boot_high':boot.quantile(.975)}))

<a id="section-09"></a>

## 09 — Mean and median by build year

Group by boat build year, not model's last-built year. Report count, mean, median and interquartile range; plot both with sparse years highlighted. Missing/invalid years are excluded and their count reported. This is cross-sectional cohort composition, not a depreciation curve: newer boats can be different sizes/models and survivorship/condition can differ. Compare within size/hull strata or fit the adjusted pricing model. Without listing dates, age assumes the configured valuation year; actual longitudinal depreciation needs repeated comparable transactions/listings and time controls.

In [ ]:
year_prices = priced.groupby('year_built').price_usd.agg(count='count',mean='mean',median='median',q25=lambda s:s.quantile(.25),q75=lambda s:s.quantile(.75)).sort_index()
display(year_prices)
print('Priced boats missing build year:',priced.year_built.isna().sum())
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(year_prices.index,year_prices['mean'],label='mean',alpha=.8)
ax.plot(year_prices.index,year_prices['median'],label='median')
ax.fill_between(year_prices.index,year_prices.q25,year_prices.q75,alpha=.15,label='interquartile range')
sparse=year_prices['count']<MIN_CATEGORY_N
ax.scatter(year_prices.index[sparse],year_prices['median'][sparse],marker='x',color='red',label='sparse years')
ax.set(xlabel='Build year',ylabel='Asking price (USD)',title='Asking price by build-year cohort')
ax.legend(); plt.tight_layout(); plt.show()
display(priced.groupby(['year_built','hull_type','size_bin'],observed=True).price_usd.agg(['count','mean','median']).head(25))

<a id="section-10"></a>

## 10 — Part 2: target, features and unseen-make splits

Target = positive asking price in USD. MAE predicts a conditional median under absolute loss; squared-dollar loss favors a conditional mean and is dominated by expensive boats. Use MAE as dealer-facing default, RMSE/log error/R² as complements; inspect percentage error by price band, since MAPE can be unstable on cheap boats. Models trained on log(price) and exponentiated produce a geometric-center estimate (a conditional median under suitable log-error symmetry), not automatically mean fair value. This code also includes a raw-dollar forest to compare objectives.

Features: physical dimensions, sail areas, engine power, displacement/headroom, age and hull/material/rig/category; optional brand/region. Exclude exact variant as a default predictive input, fine location/designer high-cardinality identifiers and last-built-year/archive data of questionable historical availability. Include brand only in a separately validated model; unknown one-hot values fall back to physical/general features. Do not target-encode using all rows. Age = reference year - year built, deck proxy, beam/length, sail-area/displacement scale ratios are deterministic features; all imputation/scaling is train-only.

Primary deployment test: whole makes held out. Secondary: whole make+variant combinations held out. If repeated listings of the same boat span model groups, deduplicate by boat ID first. A random row split would allow specification memorization. GroupKFold estimates cold-start behavior but cannot guarantee reliable prices for arbitrary boats far outside observed size/age/type ranges.

In [ ]:
def engineer(frame):
    out=frame.copy()
    out['age']=REFERENCE_YEAR-out.year_built
    out['deck_area_m2']=out.hull_length_m*out.beam_m
    out['beam_length_ratio']=out.beam_m/out.hull_length_m
    # displacement^(2/3) is a scale relation, not a dimensionless SI ratio without density
    out['sail_displacement_scale']=out.upwind_sail_area_m2/(out.light_displacement_kg**(2/3))
    out['power_per_tonne']=out.engine_power_kw/(out.light_displacement_kg/1000)
    out.replace([np.inf,-np.inf],np.nan,inplace=True)
    return out
model_data=engineer(priced)
model_data=model_data[model_data.make.notna() & model_data.variant.notna()].copy().reset_index(drop=True)
print('Exclude missing make/variant from group validation:',len(priced)-len(model_data))
assert model_data.make.nunique()>=6, 'Default CV needs >=6 makes; adapt folds/report limitation'
NUM = ['hull_length_m','beam_m','draft_m','upwind_sail_area_m2','downwind_sail_area_m2','mainsail_area_m2',
       'engine_power_kw','light_displacement_kg','headroom_m','age','deck_area_m2','beam_length_ratio','sail_displacement_scale','power_per_tonne']
GENERAL_CAT = ['hull_type','category','construction','appendages','rigging_type']
for c in NUM+GENERAL_CAT+['make','geographic_region']:
    if c not in model_data: model_data[c]=np.nan
train_i,test_i = next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(model_data,groups=model_data.make))
development=model_data.iloc[train_i].reset_index(drop=True)
holdout=model_data.iloc[test_i].reset_index(drop=True)
assert set(development.make).isdisjoint(set(holdout.make))
assert development.make.nunique()>=3
CV_MAKE=list(GroupKFold(n_splits=min(4,development.make.nunique())).split(development,groups=development.make))
CV_VARIANT=list(GroupKFold(n_splits=min(4,development.variant_key.nunique())).split(development,groups=development.variant_key))
print('Development rows/makes:',len(development),development.make.nunique())
print('Reserved rows/unseen makes:',len(holdout),holdout.make.nunique())
display(pd.DataFrame([{'fold':k,'train_rows':len(a),'valid_rows':len(b),'unseen_makes':development.iloc[b].make.nunique()} for k,(a,b) in enumerate(CV_MAKE)]))

<a id="section-11"></a>

## 11 — Baselines and pricing models

Compare a raw median baseline, interpretable log-Ridge, log-Random Forest and a raw-dollar Random Forest. Ridge has smooth scale effects and can extrapolate but may underfit nonlinear relationships; trees capture interactions but cannot extrapolate usefully beyond training support. Compare a physical-only/general model to optional brand+region, so cold-start behavior and geographic contribution are visible. Keep one-hot cardinality bounded. Dense encoding is intended for modest tabular challenges; switch to sparse/compatible estimators or CatBoost if available for large data. Unknown categories are handled, not declared familiar. No claims about the best model precede actual CV.

In [ ]:
def preprocess_for(cats):
    options={'handle_unknown':'ignore'}
    options['sparse_output' if 'sparse_output' in inspect.signature(OneHotEncoder).parameters else 'sparse']=False
    return ColumnTransformer([
        ('numeric',Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),NUM),
        ('category',Pipeline([('impute',SimpleImputer(strategy='constant',fill_value='__MISSING__')),('onehot',OneHotEncoder(**options))]),cats)],remainder='drop')
def pricing_pipeline(model,cats=GENERAL_CAT,log_target=True):
    pipe=Pipeline([('preprocess',preprocess_for(cats)),('model',model)])
    return TransformedTargetRegressor(regressor=pipe,func=np.log,inverse_func=np.exp) if log_target else pipe
forest_args=dict(n_estimators=120,min_samples_leaf=5,max_features=.8,random_state=SEED,n_jobs=1)
models={
 'median_baseline':DummyRegressor(strategy='median'),
 'log_ridge_general':pricing_pipeline(Ridge(alpha=10)),
 'log_forest_general':pricing_pipeline(RandomForestRegressor(**forest_args)),
 'log_forest_brand_region':pricing_pipeline(RandomForestRegressor(**forest_args),GENERAL_CAT+['make','geographic_region']),
 'raw_forest_general':pricing_pipeline(RandomForestRegressor(**forest_args),log_target=False)}
ALL_FEATURES=NUM+GENERAL_CAT+['make','geographic_region']
X=development[ALL_FEATURES].copy(); y=development.price_usd.copy()
for c in GENERAL_CAT+['make','geographic_region']:
    X[c]=X[c].astype(object).where(X[c].notna(),np.nan)
def score_price(actual,pred):
    pred=np.maximum(np.asarray(pred,dtype=float),.01)
    actual=np.asarray(actual,dtype=float)
    return {'mae_usd':mean_absolute_error(actual,pred),'rmse_usd':np.sqrt(mean_squared_error(actual,pred)),
            'r2':r2_score(actual,pred),'rms_log_error':np.sqrt(np.mean((np.log(actual)-np.log(pred))**2)),
            'median_abs_pct_error':100*np.median(np.abs(pred/actual-1))}

<a id="section-12"></a>

## 12 — Compare make and variant validation

Print fold distributions and both row-weighted and make-weighted errors; a make with many listings should not be mistaken for universal performance. Select on development unseen-make MAE and use complexity/stability to resolve near ties. Variant folds answer a different deployment question and may incidentally contain unseen makes too; report the actual fraction. Do not tune using the reserved holdout. Fold training sets overlap, so fold SD is descriptive and does not form an ordinary iid confidence interval. Selection among these models still induces optimism in the CV winner; the held-out group test or a separate dataset is required for stronger evidence.

In [ ]:
results=[]; pooled_results=[]; OOF={}; fitted_folds={}
for validation,folds in [('unseen_make',CV_MAKE),('unseen_variant',CV_VARIANT)]:
    for name,candidate in models.items():
        predictions=np.full(len(X),np.nan); fitted_folds[(validation,name)]=[]
        for k,(a,b) in enumerate(folds):
            estimator=clone(candidate).fit(X.iloc[a],y.iloc[a])
            pred=estimator.predict(X.iloc[b]); predictions[b]=pred
            metrics=score_price(y.iloc[b],pred)
            macro=pd.DataFrame({'make':development.iloc[b].make.values,'error':np.abs(y.iloc[b].values-pred)}).groupby('make').error.mean().mean()
            results.append(dict(validation=validation,model=name,fold=k,mae_make_macro=macro,
                fraction_unseen_make=float((~development.iloc[b].make.isin(development.iloc[a].make)).mean()),**metrics))
            fitted_folds[(validation,name)].append(estimator)
        OOF[(validation,name)]=predictions
        macro_all=pd.DataFrame({'make':development.make,'error':np.abs(y-predictions)}).groupby('make').error.mean().mean()
        pooled_results.append(dict(validation=validation,model=name,mae_make_macro=macro_all,**score_price(y,predictions)))
fold_results=pd.DataFrame(results)
comparison=fold_results.groupby(['validation','model']).agg(mae_usd_mean=('mae_usd','mean'),mae_usd_sd=('mae_usd','std'),mae_make_macro=('mae_make_macro','mean'),rms_log_error=('rms_log_error','mean'))
display(comparison,fold_results[['validation','model','fold','fraction_unseen_make']])
pooled_comparison=pd.DataFrame(pooled_results).set_index(['validation','model'])
display(pooled_comparison)
primary=pooled_comparison.loc['unseen_make'].sort_values('mae_usd')
selected=primary.index[0]
print('Provisional model by unseen-make MAE:',selected)

<a id="section-13"></a>

## 13 — Residuals, segments, robustness and feature importance

Inspect unseen-make out-of-fold predictions: dollar/log residuals, actual vs predicted, error vs size/age/price band and group. Compare selected-minus-baseline losses by whole-make bootstrap, interpreting it as conditional on fitted predictions and model selection. Permute original raw feature columns on validation rows; this groups one-hot representations correctly. Strongly correlated size measures substitute, so individual importance is ambiguous; optional grouped-size permutation and feature ablation test incremental value. Importance is predictive, not causal. If baseline wins, say so and skip meaningless importance rather than interpreting arbitrary features.

In [ ]:
pred=OOF[('unseen_make',selected)]
diagnostic=development[['row_id','make','variant','hull_type','geographic_region','hull_length_m','age','price_usd']].copy()
diagnostic['prediction']=pred
diagnostic['error']=diagnostic.price_usd-pred
diagnostic['abs_error']=diagnostic.error.abs()
diagnostic['abs_pct_error']=100*diagnostic.abs_error/diagnostic.price_usd
diagnostic['log_error']=np.log(diagnostic.price_usd)-np.log(np.maximum(pred,.01))
fig,axes=plt.subplots(1,3,figsize=(14,4))
axes[0].scatter(diagnostic.price_usd,pred,s=12,alpha=.5); axes[0].set(xscale='log',yscale='log',xlabel='Actual USD',ylabel='Predicted USD',title='Unseen-make OOF predictions')
axes[1].scatter(pred,diagnostic.error,s=12,alpha=.5); axes[1].axhline(0,color='black'); axes[1].set(xlabel='Predicted USD',ylabel='Dollar residual',title='Error scale/misspecification')
axes[2].hist(diagnostic.log_error,bins=30); axes[2].set(xlabel='Log residual',title='Multiplicative error distribution')
plt.tight_layout(); plt.show()
for c in ['make','hull_type','geographic_region']:
    display(diagnostic.groupby(c,dropna=False).agg(n=('abs_error','count'),mae_usd=('abs_error','mean'),median_abs_pct=('abs_pct_error','median')).sort_values('mae_usd',ascending=False).head(15))
diagnostic['price_band']=pd.qcut(diagnostic.price_usd,4,duplicates='drop')
display(diagnostic.groupby('price_band',observed=True).agg(n=('abs_error','count'),mae_usd=('abs_error','mean'),median_abs_pct=('abs_pct_error','median')))
base=OOF[('unseen_make','median_baseline')]
gain=pd.DataFrame({'make':development.make,'loss_gain':np.abs(y-base)-np.abs(y-pred)})
cluster_gains=gain.groupby('make').loss_gain.mean().values
random=np.random.RandomState(SEED)
boot_gain=[random.choice(cluster_gains,len(cluster_gains),replace=True).mean() for _ in range(500)]
print('Mean make-level MAE improvement vs baseline:',cluster_gains.mean())
print('Conditional cluster-bootstrap interval:',np.quantile(boot_gain,[.025,.975]))

importance_tables=[]
if RUN_IMPORTANCE and selected!='median_baseline':
    for k,(a,b) in enumerate(CV_MAKE):
        fitted=fitted_folds[('unseen_make',selected)][k]
        imp=permutation_importance(fitted,X.iloc[b],y.iloc[b],scoring='neg_mean_absolute_error',n_repeats=3,random_state=SEED,n_jobs=1)
        importance_tables.append(pd.DataFrame({'feature':ALL_FEATURES,'mae_increase_usd':imp.importances_mean,'fold':k}))
    importance_summary=pd.concat(importance_tables).groupby('feature').mae_increase_usd.agg(['mean','std']).sort_values('mean',ascending=False)
    display(importance_summary)
    importance_summary['mean'].head(12).sort_values().plot.barh(title='Validation permutation importance')
    plt.xlabel('MAE increase (USD)'); plt.tight_layout(); plt.show()
if RUN_ABLATION and selected!='median_baseline':
    # Assess whether optional brand/region add value using the already-computed models.
    display(comparison.loc['unseen_make'].loc[['log_forest_general','log_forest_brand_region']])
    # Group-permute correlated dimensions using one shared row permutation.
    a,b=CV_MAKE[-1]; fitted=fitted_folds[('unseen_make',selected)][-1]
    altered=X.iloc[b].copy(); order=np.random.RandomState(SEED).permutation(len(b))
    size_columns=['hull_length_m','beam_m','draft_m','deck_area_m2','upwind_sail_area_m2','downwind_sail_area_m2','mainsail_area_m2','light_displacement_kg','beam_length_ratio','sail_displacement_scale','power_per_tonne']
    altered[size_columns]=altered[size_columns].iloc[order].to_numpy()
    print('Grouped-size shuffle MAE increase:',mean_absolute_error(y.iloc[b],fitted.predict(altered))-mean_absolute_error(y.iloc[b],fitted.predict(X.iloc[b])))

<a id="section-14"></a>

## 14 — Freeze and reserved unseen-make evaluation

RUN_HOLDOUT is off by default. Freeze features/cleaning/model before enabling it. The test makes are absent from all training rows. Compare baseline and selected model, segment the test errors and record make identities/counts. If earlier full-dataset EDA influenced your choices, explicitly report this as an internal benchmark with possible selection contamination, not a pristine independent test. Report observed out-of-support features separately; handling unknown categories does not solve domain shift.

In [ ]:
final_development_model=clone(models[selected]).fit(X,y)
if RUN_HOLDOUT:
    Xh=holdout[ALL_FEATURES].copy()
    for c in GENERAL_CAT+['make','geographic_region']:
        Xh[c]=Xh[c].astype(object).where(Xh[c].notna(),np.nan)
    holdout_pred=final_development_model.predict(Xh)
    baseline=clone(models['median_baseline']).fit(X,y)
    display(pd.DataFrame([dict(model=selected,**score_price(holdout.price_usd,holdout_pred)),dict(model='median_baseline',**score_price(holdout.price_usd,baseline.predict(Xh)))]))
    display(pd.DataFrame({'make':holdout.make,'abs_error':np.abs(holdout.price_usd-holdout_pred)}).groupby('make').abs_error.agg(['count','mean']))

<a id="section-15"></a>

## 15 — Predict ten boats and report average/median

With TEN_BOATS_PATH supplied, reuse exactly the same schema mapping, deterministic cleaning and units, then predict with the development-fitted model. Refit on all labeled data only after the evaluation and model choice are finalized (explicitly do so yourself). Never include price in predictive features. No file supplied: select ten reserved boats using features/seed only and illustrate comparison to held-out actual prices. They must not be used for tuning. Report individual prices, total, mean and median across the ten point predictions; these are not estimates of the dealer's unknown aggregate realized sales. Flag unseen makes/variants and numeric extrapolation/missingness. Prediction intervals require a separate group-aware calibration protocol; ordinary iid conformal guarantees do not automatically apply to new manufacturers.

In [ ]:
if TEN_BOATS_PATH is not None:
    raw_ten=pd.read_csv(TEN_BOATS_PATH)
    ten,ten_map=canonicalize(raw_ten)
    ten,ten_parse_audit=clean_frame(ten)
    ten=clean_categories(ten)
    # Same deterministic invalid-value treatment; do NOT impute from ten boats.
    for c in UNIT_DEFAULT:
        invalid=ten[c].notna() & ((~np.isfinite(ten[c])) | (ten[c]<=0))
        ten.loc[invalid,c]=np.nan
    for c in ['year_built','last_built_year']:
        ten.loc[(ten[c]<1800)|(ten[c]>REFERENCE_YEAR)|(ten[c]%1!=0),c]=np.nan
    ten=engineer(ten)
    display(ten_parse_audit[ten_parse_audit.status.str.contains('ambiguous|conflicting|unparsed|wrong')].head(20))
else:
    assert len(holdout)>=10
    ten=holdout.sample(n=10,random_state=SEED).copy()
    print('Illustration: ten reserved rows; not the unspecified challenge boats')
assert len(ten)==10, 'This question requests exactly ten boats'
ten_X=ten[ALL_FEATURES].copy()
for c in GENERAL_CAT+['make','geographic_region']:
    ten_X[c]=ten_X[c].astype(object).where(ten_X[c].notna(),np.nan)
ten_prices=final_development_model.predict(ten_X)
ten_predictions=ten[['make','variant']].copy().reset_index(drop=True)
ten_predictions['predicted_price_usd']=ten_prices
ten_predictions['unseen_make']=~ten['make'].isin(development.make).to_numpy()
known_variants=set(development.make+'::'+development.variant)
ten_predictions['unseen_variant']=[str(a)+'::'+str(b) not in known_variants for a,b in zip(ten.make,ten.variant)]
ten_predictions['missing_numeric_fraction']=ten[NUM].isna().mean(axis=1).to_numpy()
outside=np.zeros(len(ten),dtype=bool)
for c in NUM:
    if development[c].notna().any():
        outside |= ((ten[c]<development[c].min())|(ten[c]>development[c].max())).to_numpy()
ten_predictions['outside_training_numeric_range']=outside
if TEN_BOATS_PATH is None:
    ten_predictions['actual_price_usd']=ten.price_usd.to_numpy()
display(ten_predictions)
ten_summary=pd.Series({'count':len(ten_prices),'total_predicted_usd':np.sum(ten_prices),
    'average_predicted_usd':np.mean(ten_prices),'median_predicted_usd':np.median(ten_prices)})
display(ten_summary)
if RUN_SAVE:
    ten_predictions.to_csv('ten_boat_predictions.csv',index=False)
    ten_summary.to_csv('ten_boat_summary.csv')

<a id="section-16"></a>

## 16 — Write the conclusions and next steps

Use the following evidence-backed answer structure after running on real data:

1. **Cleaning:** enumerate raw unit/type issues, parse failures, currency checks, invalid years, duplicate treatment and deck-proxy discrepancies with counts. Explain that unresolved values remain missing and model imputation is fold-specific.
2. **Q1a:** give Beneteau average mainsail area (m²) plus sample count/missingness; highest-median category draft (m) plus ties/count and robustness.
3. **Q3 correlations:** name top three unique pairs with signed Pearson r, Spearman r and n; explain mechanical dependence for deck proxy and repeated model specifications. Describe price tail and top-quantile patterns with comparisons.
4. **Region/year:** separate raw composition effects from adjusted associations, discuss overlap/confounders and instability. Build-year trends are cross-sectional, not measured depreciation.
5. **Pricing:** state asking-price target, physical/general features and excluded identifiers; show baseline vs models under unseen-make and unseen-variant evaluation. Explain model objective, unknown-category fallback, error scale and uncertainty.
6. **Ten boats:** present ten predictions, flags, mean/median/total; state whether they were provided boats or an illustrative held-out sample.

Limitations: listing price vs completed sale price, omitted condition/refits/equipment/engine hours, listing time/inflation/FX and regional tax differences, dataset selection/survivorship, duplicate ads, model-size extrapolation and few independent brands. A useful dealer model should add condition/equipment/sales history, calibrate against realized transaction prices, use group-aware uncertainty and abstain/refer to expert appraisal for unfamiliar boats outside support.

Expected hypotheses are not findings: size/newness may predict prices; deck/length/beam may be highly correlated; raw regions may reflect composition. Fill numerical claims from actual outputs only. If the unseen-make baseline wins, report failure to justify a more complex model.